# Example: Vaccine Response Analysis (GSE171964)
### Dataset: BNT162b2 vaccination time course (GEO GSE171964)


## Background
This dataset profiles PBMCs collected at baseline and after BNT162b2 vaccination, capturing short‑term immune dynamics.

### Important Methodological Notes
**Single‑arm longitudinal study:**
- All participants received vaccine
- Analyses focus on **within‑participant** changes over time
- We do not estimate treatment effects versus an unvaccinated control

**Time axis:**
- We use **Day 0 vs Day 7** as the primary comparison

**Analysis strategy:**
- Participant‑level aggregation avoids treating cells as independent
- Cross‑sectional summaries are descriptive; inference uses paired comparisons
- Multiple testing is controlled using FDR (alpha = 0.25)

**Sampling and scope:**
- We restrict to participants with both Day 0 and Day 7 samples
- For speed, we cap the number of participants and cells per participant/day/cell type


## 1. Setup


In [1]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import sctrial as st
import scipy.sparse as sp
from statsmodels.stats.multitest import multipletests

pd.options.mode.chained_assignment = None

# Configuration
SEED = 42
MIN_PAIRED = 4
MIN_GENES_FOR_SCORE = 5
FDR_ALPHA = 0.25

sc.settings.set_figure_params(dpi=120)


## 2. Data Loading and Processing


In [2]:
from pathlib import Path
import gzip
from scipy.io import mmread


def _resolve_dir_with_files(p: str, required_files) -> Path:
    path = Path(p)
    if path.is_absolute():
        if all((path / f).exists() for f in required_files):
            return path
    for base in [Path.cwd(), *Path.cwd().parents]:
        cand = base / path
        if all((cand / f).exists() for f in required_files):
            return cand
    return path


def _params_match(prev, curr):
    for k, v in curr.items():
        pv = prev.get(k, None)
        if isinstance(pv, (list, tuple)):
            pv_list = list(pv)
        elif hasattr(pv, 'tolist'):
            pv_list = pv.tolist()
        else:
            pv_list = pv
        if isinstance(v, (list, tuple)):
            if list(v) != pv_list:
                return False
        else:
            if pv_list != v:
                return False
    return True


def load_vaccine_example_data(
    data_dir="data/vaccine_gse171964",
    processed_name="vaccine_gse171964_day0_day7.h5ad",
    max_participants=30,
    max_cells_per_group=200,
    seed=SEED,
    force_reprocess=False,
):
    """
    Load and preprocess GSE171964 PBMC vaccine time course data.

    We keep Day 0 and Day 7 samples, then restrict to participants with both days.
    For speed, we cap the number of participants and cells per participant/day/cell type.
    """
    data_dir = _resolve_dir_with_files(
        data_dir,
        [
            "GSE171964_barcodes_v2.tsv.gz",
            "GSE171964_feats_v2.tsv.gz",
            "GSE171964_geo_pheno_v2.csv.gz",
            "GSE171964_countsmatrix_v2.mtx.gz",
        ],
    )
    processed_path = data_dir.parent / "processed" / processed_name

    processing_params = {
        "version": "v2",
        "max_participants": max_participants,
        "max_cells_per_group": max_cells_per_group,
        "seed": seed,
        "days": [0, 7],
    }

    if processed_path.exists() and not force_reprocess:
        adata = sc.read_h5ad(processed_path)
        prev = adata.uns.get("processing_params", {})
        if _params_match(prev, processing_params):
            print(f"Loaded processed vaccine dataset (GSE171964): {adata.n_obs} cells, {adata.n_vars} genes")
            print("Processed file:", processed_path)
            print("Days:", adata.obs["day"].unique())
            print("Participants:", adata.obs["pt_id"].nunique())
            print("Cell types:", adata.obs["clustnm"].nunique())
            return adata
        else:
            print("Processed file parameters differ; reprocessing.")

    barcodes_path = data_dir / "GSE171964_barcodes_v2.tsv.gz"
    feats_path = data_dir / "GSE171964_feats_v2.tsv.gz"
    pheno_path = data_dir / "GSE171964_geo_pheno_v2.csv.gz"
    mtx_path = data_dir / "GSE171964_countsmatrix_v2.mtx.gz"

    for p in [barcodes_path, feats_path, pheno_path, mtx_path]:
        if not p.exists():
            raise FileNotFoundError(f"Missing file: {p}")

    barcodes = (
        pd.read_csv(barcodes_path, sep="\s+", header=None, engine="python", skiprows=1)[1]
        .astype(str)
        .str.strip('"')
        .tolist()
    )
    features = (
        pd.read_csv(feats_path, sep="\s+", header=None, engine="python", skiprows=1)[1]
        .astype(str)
        .str.strip('"')
        .tolist()
    )

    with gzip.open(mtx_path, "rb") as f:
        X = mmread(f).tocsr()

    if X.shape[0] == len(features) and X.shape[1] == len(barcodes):
        X = X.T
    elif X.shape[0] == len(barcodes) and X.shape[1] == len(features):
        pass
    else:
        raise ValueError("Matrix dimensions do not match barcodes/features.")

    adata = sc.AnnData(X=X)
    adata.obs_names = barcodes
    adata.var_names = features

    pheno = pd.read_csv(pheno_path)
    pheno["barcode"] = pheno["barcode"].astype(str)
    pheno = pheno.set_index("barcode")
    pheno = pheno.loc[adata.obs_names]
    adata.obs = pheno

    # Keep baseline and day 7
    adata = adata[adata.obs["day"].isin([0, 7])].copy()

    # Keep paired participants and cap participants
    paired = adata.obs.groupby("pt_id")["day"].nunique()
    keep_ids = paired[paired >= 2].index
    adata = adata[adata.obs["pt_id"].isin(keep_ids)].copy()

    rng = np.random.default_rng(seed)
    uniq_ids = adata.obs["pt_id"].unique()
    n = min(len(uniq_ids), max_participants)
    sel = rng.choice(uniq_ids, size=n, replace=False)
    adata = adata[adata.obs["pt_id"].isin(sel)].copy()

    # Cap cells per participant/day/celltype
    if max_cells_per_group is not None:
        grp = ["pt_id", "day", "clustnm"]
        sampled = (
            adata.obs.groupby(grp, observed=True, group_keys=False)
            .apply(lambda x: x.sample(min(len(x), max_cells_per_group), random_state=seed))
        )
        adata = adata[sampled.index].copy()

    # Add counts layer
    adata.layers["counts"] = adata.X.copy()

    adata.uns["processing_params"] = processing_params

    processed_path.parent.mkdir(parents=True, exist_ok=True)
    adata.write_h5ad(processed_path)
    print("Saved processed file:", processed_path)

    print(f"Loaded vaccine dataset (GSE171964): {adata.n_obs} cells, {adata.n_vars} genes")
    print("Days:", adata.obs["day"].unique())
    print("Participants:", adata.obs["pt_id"].nunique())
    print("Cell types:", adata.obs["clustnm"].nunique())

    return adata


### 2.1 Load processed AnnData


In [3]:
adata = load_vaccine_example_data(
    max_participants=30,
    max_cells_per_group=200,
    seed=SEED,
)

print(adata)
print("Obs columns:")
print(sorted(adata.obs.columns.tolist()))


FileNotFoundError: Missing file: data/vaccine_gse171964/GSE171964_barcodes_v2.tsv.gz

### 2.2 Quick exploratory summaries


In [4]:
print("Days:", adata.obs["day"].unique())
print("Participants:", adata.obs["pt_id"].nunique())
print("Cell types:", adata.obs["clustnm"].nunique())

# Participants per day (before pairing restriction for analysis)
pt_per_day = adata.obs.groupby("day")["pt_id"].nunique().sort_index()
print("Participants per day:")
print(pt_per_day)

# Cell counts
eday_counts = adata.obs["day"].value_counts().sort_index()
ct_counts = adata.obs["clustnm"].value_counts().head(15)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
eday_counts.plot(kind="bar", ax=axes[0], title="Cells by day")
ct_counts.plot(kind="bar", ax=axes[1], title="Top cell types")
plt.tight_layout(); plt.show()


NameError: name 'adata' is not defined

## 3. Trial Design and Timepoint Strategy


In [5]:
# Ensure log1p CPM
if "log1p_cpm" not in adata.layers:
    adata = st.add_log1p_cpm_layer(adata, counts_layer="counts", out_layer="log1p_cpm")

# Standardize metadata
adata.obs["visit"] = adata.obs["day"].astype(str)
adata.obs["participant_id"] = adata.obs["pt_id"].astype(str)
adata.obs["arm"] = "Vaccinated"
adata.obs["cell_type"] = adata.obs["clustnm"].astype(str)

visit_col = "visit"
visits = [v for v in ["0", "7"] if v in adata.obs[visit_col].unique()]

# Participant-level pairing
participant_summary = (
    adata.obs.groupby("participant_id")[visit_col].apply(set).reset_index()
)
participant_summary["has_day0"] = participant_summary[visit_col].apply(lambda x: "0" in x)
participant_summary["has_day7"] = participant_summary[visit_col].apply(lambda x: "7" in x)
participant_summary["is_paired"] = participant_summary["has_day0"] & participant_summary["has_day7"]
paired_ids = set(participant_summary.loc[participant_summary["is_paired"], "participant_id"])

n_paired = len(paired_ids)
print(f"Paired participants (0 vs 7): {n_paired}")

# Analysis subset: paired participants only
adata_analysis = adata[adata.obs["participant_id"].isin(paired_ids)].copy()
print(f"Analysis subset cells: {adata_analysis.n_obs}")

# Covariates (if available)
candidate_covariates = ["age", "sex", "batch", "site"]
covariates = [c for c in candidate_covariates if c in adata.obs.columns]
print("Covariates available (not modeled here):", covariates)

# Design object (single arm); arm labels are placeholders for API consistency
# We rely on paired within-participant tests, not treated vs control inference.
design = st.TrialDesign(
    participant_col="participant_id",
    visit_col=visit_col,
    arm_col="arm",
    arm_treated="Vaccinated",
    arm_control="Vaccinated",
    celltype_col="cell_type",
)


NameError: name 'adata' is not defined

### 3.1 Exploratory plots of cell types per visit


In [6]:
ct_visit = (
    adata_analysis.obs
    .groupby([design.celltype_col, design.visit_col], observed=True)
    .size()
    .reset_index(name="n_cells")
)

top_ct = adata_analysis.obs[design.celltype_col].value_counts().head(10).index
ct_visit_top = ct_visit[ct_visit[design.celltype_col].isin(top_ct)].copy()

plt.figure(figsize=(12, 5))
sns.barplot(data=ct_visit_top, x=design.celltype_col, y="n_cells", hue=design.visit_col)
plt.title("Top cell types by visit (paired participants)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout(); plt.show()


NameError: name 'adata_analysis' is not defined

## 4. UMAPs of All Cell Types


In [7]:
# Compute UMAP on log1p CPM if missing
adata_umap = adata_analysis.copy()
adata_umap.X = adata_umap.layers["log1p_cpm"].copy()

if "X_umap" not in adata_umap.obsm:
    sc.pp.pca(adata_umap)
    sc.pp.neighbors(adata_umap)
    sc.tl.umap(adata_umap)

sc.pl.umap(
    adata_umap,
    color=design.celltype_col,
    legend_loc="right margin",
    title="All cell types (paired participants)",
)

# Stratified UMAPs by visit
fig, axes = plt.subplots(1, len(visits), figsize=(16, 4.5))
if len(visits) == 1:
    axes = [axes]
for i, vis in enumerate(visits):
    sub = adata_umap[adata_umap.obs[design.visit_col] == vis].copy()
    ax = axes[i]
    if sub.n_obs > 0:
        sc.pl.umap(sub, color=design.celltype_col, ax=ax, show=False, legend_loc=None, frameon=False)
        ax.set_title(f"Day {vis}")
        ax.set_aspect('equal')
    else:
        ax.set_axis_off()
plt.tight_layout(); plt.show()


NameError: name 'adata_analysis' is not defined

## 5. Module Scoring and Gene Panel


In [8]:
available = set(adata_analysis.var_names)

# Define immunologically relevant gene sets
raw_gene_sets = {
    "B_Cell_Response": ["CD79A", "CD79B", "MS4A1", "MZB1", "XBP1", "JCHAIN"],
    "IFN_Signature": ["ISG15", "IFI6", "IFIT1", "IFIT3", "MX1", "STAT1", "OAS1"],
    "Inflammatory": ["S100A8", "S100A9", "LYZ", "VCAN", "IL1B", "CXCL8"],
}

print("Gene set coverage:")
filtered = {}
for name, genes in raw_gene_sets.items():
    present = [g for g in genes if g in available]
    pct = len(present) / len(genes) * 100
    status = "OK" if len(present) >= MIN_GENES_FOR_SCORE else "SKIP"
    print(f"  {name}: {len(present)}/{len(genes)} genes ({pct:.0f}%) [{status}]")
    if len(present) >= MIN_GENES_FOR_SCORE:
        filtered[name] = present

if filtered:
    adata_analysis = st.score_gene_sets(
        adata_analysis, filtered, layer="log1p_cpm", method="zmean", prefix="ms_"
    )
    print(f"Scored {len(filtered)} gene sets using zmean.")
else:
    print(f"No gene sets matched (min_genes={MIN_GENES_FOR_SCORE}); skipping module scoring.")

features = [c for c in adata_analysis.obs.columns if c.startswith("ms_")]
print("Module score features:", features)

panel_genes_all = [
    "CD79A", "CD79B", "MS4A1", "MZB1", "XBP1",
    "ISG15", "IFI6", "IFIT1", "MX1",
    "S100A8", "S100A9", "LYZ", "VCAN",
]
panel_genes = [g for g in panel_genes_all if g in available]
missing_panel = [g for g in panel_genes_all if g not in available]
print("Panel genes (available):", panel_genes)
if missing_panel:
    print("Panel genes missing:", missing_panel)


NameError: name 'adata_analysis' is not defined

## 6. Within‑Arm Comparisons (Participant‑Level)


In [9]:
def _ensure_fdr(df, p_col="p_time", fdr_col="FDR_time"):
    if df.empty:
        return df
    if fdr_col in df.columns:
        return df
    if p_col in df.columns:
        mask = df[p_col].notna()
        df[fdr_col] = np.nan
        if mask.sum() > 0:
            df.loc[mask, fdr_col] = multipletests(df.loc[mask, p_col], method="fdr_bh")[1]
    return df

if features and n_paired >= MIN_PAIRED and len(visits) == 2:
    res_within = st.within_arm_comparison(
        adata_analysis,
        arm="Vaccinated",
        features=features,
        design=design,
        visits=tuple(visits),
        aggregate="participant_visit",
    )
    print("Within-arm (participant-level module scores):")
    if not res_within.empty:
        res_within = _ensure_fdr(res_within)
        display(res_within.set_index("feature"))
        if "FDR_time" in res_within.columns:
            sig = res_within[res_within["FDR_time"] < FDR_ALPHA]
            if not sig.empty:
                print(f"Significant (FDR < {FDR_ALPHA}):", sig["feature"].tolist())
else:
    print("Insufficient paired participants for within-arm comparisons.")


NameError: name 'features' is not defined

## 7. Gene Expression Within‑Arm (Panel Genes)


In [10]:
def _ensure_fdr(df, p_col="p_time", fdr_col="FDR_time"):
    if df.empty:
        return df
    if fdr_col in df.columns:
        return df
    if p_col in df.columns:
        mask = df[p_col].notna()
        df[fdr_col] = np.nan
        if mask.sum() > 0:
            df.loc[mask, fdr_col] = multipletests(df.loc[mask, p_col], method="fdr_bh")[1]
    return df

if panel_genes and n_paired >= MIN_PAIRED and len(visits) == 2:
    res_genes = st.within_arm_comparison(
        adata_analysis,
        arm="Vaccinated",
        features=panel_genes,
        design=design,
        visits=tuple(visits),
        aggregate="participant_visit",
        layer="log1p_cpm",
    )
    print("Within-arm (panel genes):")
    if not res_genes.empty:
        res_genes = _ensure_fdr(res_genes)
        display(res_genes.set_index("feature"))
        if "FDR_time" in res_genes.columns:
            sig = res_genes[res_genes["FDR_time"] < FDR_ALPHA]
            if not sig.empty:
                print(f"Significant (FDR < {FDR_ALPHA}):", sig["feature"].tolist())
else:
    print("Insufficient paired participants for gene-level comparisons.")


NameError: name 'panel_genes' is not defined

## 8. Pseudobulk Within‑Arm (Panel Genes)


In [11]:
if panel_genes and n_paired >= MIN_PAIRED and len(visits) == 2:
    Xc = adata_analysis.layers["counts"]
    if sp.issparse(Xc):
        Xc = Xc.tocsr()

    total_counts = np.asarray(Xc.sum(axis=1)).ravel()
    gene_idx = [adata_analysis.var_names.get_loc(g) for g in panel_genes]
    X_panel = Xc[:, gene_idx]
    if sp.issparse(X_panel):
        X_panel = X_panel.toarray()

    df_expr = pd.DataFrame(X_panel, columns=panel_genes, index=adata_analysis.obs_names)
    df_meta = adata_analysis.obs[["participant_id", design.visit_col, design.celltype_col]].copy()
    df_meta["total_counts"] = total_counts
    df = df_meta.join(df_expr, how="left")

    df_sum = (
        df.groupby(["participant_id", design.visit_col, design.celltype_col], observed=True)
          .sum(numeric_only=True)
          .reset_index()
    )

    totals = df_sum["total_counts"].values.reshape(-1, 1)
    cpm = df_sum[panel_genes].values / (totals + 1e-12) * 1e6
    df_sum[panel_genes] = np.log1p(cpm)

    # compute paired deltas per cell type
    rows = []
    for ct in df_sum[design.celltype_col].unique():
        sub = df_sum[df_sum[design.celltype_col] == ct].copy()
        wide = sub.pivot_table(index="participant_id", columns=design.visit_col, aggfunc="size", fill_value=0, observed=True)
        keep = wide[(wide.get(visits[0], 0) > 0) & (wide.get(visits[1], 0) > 0)].index
        sub = sub[sub["participant_id"].isin(keep)].copy()
        if sub["participant_id"].nunique() < MIN_PAIRED:
            continue
        for g in panel_genes:
            w = sub.pivot_table(index="participant_id", columns=design.visit_col, values=g, aggfunc="mean")
            if visits[0] not in w.columns or visits[1] not in w.columns:
                continue
            delta = (w[visits[1]] - w[visits[0]]).dropna()
            rows.append({
                "celltype": ct,
                "feature": g,
                "n_units": int(len(delta)),
                "mean_delta": float(delta.mean()),
                "median_delta": float(delta.median()),
            })

    pb = pd.DataFrame(rows)
    print("Pseudobulk paired deltas (panel genes × cell type):")
    if not pb.empty:
        display(pb.sort_values(["celltype", "feature"]).head(50))
else:
    print("Insufficient paired participants for pseudobulk.")


NameError: name 'panel_genes' is not defined

## 9. Trial Interaction Plot (Single Arm)


In [12]:
if features and len(visits) == 2:
    fig, ax = plt.subplots(1, 1, figsize=(5, 4))
    st.plot_within_arm_comparison(
        adata_analysis,
        arm="Vaccinated",
        feature=features[0],
        design=design,
        visits=tuple(visits),
        plot_type="paired",
        ax=ax,
    )
    plt.tight_layout(); plt.show()


NameError: name 'features' is not defined

## 10. Trial UMAP Panel for a Module Score


In [13]:
if features:
    from matplotlib.gridspec import GridSpec

    feature = features[0]
    ad = adata_analysis.copy()
    ad.X = ad.layers["log1p_cpm"].copy()

    if "X_umap" not in ad.obsm:
        sc.pp.pca(ad)
        sc.pp.neighbors(ad)
        sc.tl.umap(ad)

    fig = plt.figure(figsize=(18, 5))
    gs = GridSpec(2, 3, figure=fig, width_ratios=[1.6, 1, 1], height_ratios=[1, 1])

    # Big cell type UMAP
    ax_big = fig.add_subplot(gs[:, 0])
    sc.pl.umap(ad, color=design.celltype_col, ax=ax_big, show=False, frameon=False, title="Cell Types")
    ax_big.set_aspect('equal')

    # Feature UMAPs by visit
    vals = ad.obs[feature].values
    vmin = np.nanpercentile(vals, 1)
    vmax = np.nanpercentile(vals, 99)

    for vis, (r, c) in [(visits[0], (0, 1)), (visits[1], (0, 2))]:
        ax = fig.add_subplot(gs[r, c])
        sub = ad[ad.obs[design.visit_col] == vis].copy()
        if sub.n_obs > 0:
            sc.pl.umap(sub, color=feature, ax=ax, show=False, frameon=False, vmin=vmin, vmax=vmax, cmap="magma", title=f"Day {vis}")
            ax.set_aspect('equal')
        else:
            ax.set_axis_off()

    # leave bottom row blank to avoid vertical stretch
    ax_blank1 = fig.add_subplot(gs[1, 1]); ax_blank1.axis('off')
    ax_blank2 = fig.add_subplot(gs[1, 2]); ax_blank2.axis('off')

    plt.tight_layout(); plt.show()


NameError: name 'features' is not defined

## 11. Dotplot of Panel Genes


In [14]:
if panel_genes:
    sc.pl.dotplot(
        adata_analysis,
        panel_genes,
        groupby=design.visit_col,
        standard_scale="var",
        use_raw=False,
    )


NameError: name 'panel_genes' is not defined